# 주문(orders) CRUD

주문은 회원(`user_id`)과 상품(`product_id`)을 외래키로 참조합니다.

In [1]:
import sys
sys.path.append("..")  # 프로젝트 루트의 shopping 패키지를 불러오기 위해

from pprint import pprint
from postgrest.exceptions import APIError
from shopping import users, products, orders

In [2]:
hong = users.get_user_by_email("hong@example.com")
keyboard = products.get_products()[0]
print("주문자:", hong["name"], "/ 상품:", keyboard["name"], keyboard["price"], "원")

주문자: 홍길동 / 상품: 무선 기계식 키보드 120000 원


## Create - 주문 추가 (주문 시점 상품명·가격 저장)

In [3]:
new_order = orders.create_order(hong["id"], keyboard["id"], quantity=2)
pprint(new_order)

{'created_at': '2026-11-14T15:28:58.229733+00:00',
 'id': '6aa59345-a080-4594-9306-d27326938c8d',
 'item_name': '무선 기계식 키보드',
 'modified_at': None,
 'order_status': 'READY',
 'price': 120000,
 'product_id': '7e5ac29e-a8a5-4852-b731-c1fd32e22e56',
 'quantity': 2,
 'total_price': 240000,
 'user_id': '6376b264-1aae-4c1c-b398-1ebf2014a0e6'}


## Read - 주문 목록 (회원 이름 함께 조회)

In [4]:
pprint(orders.get_orders(hong["id"]))

[{'id': '6aa59345-a080-4594-9306-d27326938c8d',
  'item_name': '무선 기계식 키보드',
  'order_status': 'READY',
  'price': 120000,
  'quantity': 2,
  'total_price': 240000,
  'users': {'name': '홍길동'}}]


## Update - 주문 상태 변경

In [5]:
updated = orders.update_order_status(new_order["id"], "PAID")
print("변경 후 상태:", updated["order_status"])

변경 후 상태: PAID


## 제약조건 검증

- 없는 회원으로 주문하면 FK 위반
- 주문이 있는 회원은 삭제할 수 없음 (`ON DELETE RESTRICT`)

In [6]:
try:
    orders.create_order("00000000-0000-0000-0000-000000000000", keyboard["id"])
except APIError as e:
    print("주문 실패:", e.code, e.message)

try:
    users.delete_user(hong["id"])
except APIError as e:
    print("회원 삭제 실패:", e.code, e.message)

주문 실패: 23503 insert or update on table "orders" violates foreign key constraint "orders_user_id_fkey"
회원 삭제 실패: 23503 update or delete on table "users" violates foreign key constraint "orders_user_id_fkey" on table "orders"


## Delete - 주문 삭제

In [7]:
print("삭제된 행 수:", orders.delete_order(new_order["id"]))
print("남은 주문:", orders.get_orders(hong["id"]))

삭제된 행 수: 1


남은 주문: []
